In [ ]:
%matplotlib inline
import os
import pandas as pd
import torch

In [ ]:
def convert_cell_coordinates(input_path, output_path=None):
    """
    将细胞坐标文件转换为标准X,Y格式
    
    参数:
    input_path: 输入文件路径
    output_path: 输出文件路径（可选，默认在相同目录创建新文件）
    """
    if output_path is None:
        dir_name, file_name = os.path.split(input_path)
        output_path = os.path.join(dir_name, f"converted_{file_name}")
    
    try:
        try:
            df = pd.read_csv(input_path)
        except UnicodeDecodeError:
            df = pd.read_csv(input_path, encoding='latin1')
        
        print(f"原始数据列名: {df.columns.tolist()}")
        print(f"原始数据前5行:\n{df.head()}")
        column_mapping = {}
        if 'normalized_x' in df.columns and 'normalized_y' in df.columns:
            column_mapping = {'normalized_x': 'X', 'normalized_y': 'Y'}
        elif 'x' in df.columns and 'y' in df.columns:
            column_mapping = {'x': 'X', 'y': 'Y'}
        elif 'X' in df.columns and 'Y' in df.columns:
            pass
        else:
            for col in df.columns:
                if 'x' in col.lower() and 'normalized' in col.lower():
                    column_mapping[col] = 'X'
                elif 'y' in col.lower() and 'normalized' in col.lower():
                    column_mapping[col] = 'Y'
            
            if len(column_mapping) != 2:
                raise ValueError("无法自动识别坐标列，请手动指定列映射")
        
        if column_mapping:
            df = df.rename(columns=column_mapping)
            print(f"已重命名列: {column_mapping}")
        
        if 'X' in df.columns and 'Y' in df.columns:
            df = df[['X', 'Y']]
        else:
            numeric_cols = df.select_dtypes(include=['number']).columns
            if len(numeric_cols) >= 2:
                df = df[numeric_cols[:2]]
                df.columns = ['X', 'Y']
                print("使用前两列数值数据作为坐标")
            else:
                raise ValueError("文件中未找到足够的数值列作为坐标")
        
        df.to_csv(output_path, index=False)
        print(f"转换完成! 新文件已保存至: {output_path}")
        print(f"转换后数据前5行:\n{df.head()}")
        
        return output_path
    
    except Exception as e:
        print(f"转换过程中出错: {str(e)}")
        return None


input_file = r"\input\coordinates.csv"

converted_file = convert_cell_coordinates(input_file)
    
if converted_file:
        print(f"转换成功! 文件路径: {converted_file}")
else:
        print("转换失败，请检查错误信息")


In [ ]:
from model import MGCL as mg

data_paths = {
    'coords': r"input\coords.csv",
    'z_local': r"input\z_local.pt",
    'regions': r"input\region_ids.pth",
    'z_fused': r"input/z_fused.csv",
    'edges': r"input\full_edges.pt"
}
dataset = mg.BioDataset(data_paths)
train_dataset, valid_dataset = mg.create_train_valid_split(dataset)
z_fused = pd.read_csv("output/z_fused.csv")
z_local = pd.read_csv("output\z_local.pt")
region_ids = torch.load(r"output\region_ids.pth")

from model import MGCL as mg
model = mg.BioContrastiveModel(z_local_dim=z_local.shape[1],z_fused_dim=z_fused.shape[1])
trainer = mg.BioContrastTrainer(model,train_dataset ,valid_dataset,device='cuda')

log_df = trainer.train(epochs=2000)

In [ ]:
from model import MGCL as mg
from torch.utils.data import DataLoader
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Mouse_cell_features = torch.load('./input/cell_features.pt')
comm_model = mg.CommunicationModule().to(device)
comm_trainer = mg.CommunicationTrainer(comm_model,None,device)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用的设备: {device}")

In [ ]:
lr_filename = f"./input/CCIlist.txt"

train_pairs, train_labels, valid_pairs, valid_labels = mg.load_and_split_cell_pairs(
    lr_filename,
    train_ratio=0.8,
    random_seed=42
)
train_comm_dataset = mg.CommDataset(
    {'cell_features': Mouse_cell_features},
    train_pairs,
    train_labels,
    batch_size=20000,
    mode='train'
)
valid_comm_dataset = mg.CommDataset(
    {'cell_features': Mouse_cell_features},
    valid_pairs,
    valid_labels,
    batch_size=20000,
    mode='valid'
)

train_comm_loader = DataLoader(train_comm_dataset, batch_size=None, shuffle=True)
valid_comm_loader = DataLoader(valid_comm_dataset, batch_size=None, shuffle=False)
epochs = 20000
comm_trainer.train(
    train_loader=train_comm_loader,
    valid_loader=valid_comm_loader,
    epochs=epochs,
    lr=2e-5,
    save_interval=100
)